<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week1/Week1_Gradient_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
#imports
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import csv

Example 1:

Assume simple linear regression (single feature):

1. Our data is then $\{y_i,x_i\}_{i=1}^n$.
2. Our model is $\hat{y}=\beta_0+\beta_1x_i$.
3. Our loss is MSE: $\mathcal{L} = \frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2$.
4. We change the algorithm to gradient descent (this is not necessary for linear regression as we have the closed form solution, but most other models require this type of optimization in practice.)

In [35]:
y = torch.tensor([2.0, 4.1, 6.0, 8.2], dtype = torch.float32) # simple data
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype = torch.float32)

def model(x, b0, b1): # function for model that gives y_hat
  return b0 + b1*x


Next, we choose starting values for the parameters. Typically we do this randomly but we will choose $\beta_0=\beta_1=0$ for now.

In [36]:
b0 = 0 # starting values for parameters
b1 = 0

def mse_loss(y, y_hat): # loss function
  errors = y - y_hat
  squared_errors = errors ** 2
  return torch.mean(squared_errors)

For gradient descent, we need the partial derivatives of the loss function with respect to the learnable parameters of the model to obtain the gradient:

$$\frac{\partial \mathcal{L}}{d \beta_0}=-\frac{2}{n}\sum_{i=1}^n(y_i-\beta_0-\beta_1x_i)=-\frac{2}{n}\sum_{i=1}^n(y_i-\hat{y}_i)$$

$$\frac{\partial \mathcal{L}}{d \beta_1}=-\frac{2}{n}\sum_{i=1}^nx_i(y_i-\beta_0-\beta_1x_i)=-\frac{2}{n}\sum_{i=1}^nx_i(y_i-\hat{y}_i)$$

We then need to be able to compute the gradient of the loss on the entire dataset (plug in $y$ and $x$).

In [37]:
def gradients(x, y, b0, b1): # function that provides gradients of the loss computed on the dataset (x,y)
  y_hat = model(x, b0, b1) # get predicted value of y
  error = y - y_hat # get error for intermediate step

  # compute gradient components
  db0 = -2 * torch.mean(error)
  db1 = -2 * torch.mean(x * error)
  return db0.item(), db1.item() # returns as python floats

We need the learning rate, or how far we "step" or update in the direction of the negative gradient (the gradient points towards the steepest ascent, so we move in the opposite direction for the steepest descent).

In [38]:
lr = 0.1 # set learning rate (0.1, 0.01, 0.001 are common depending on application)

We then iterate over the entire dataset once per epoch, updating our parameters by the learning rate times the negative gradient each time.  So 50 epochs = 50 updates in this case.  

In [39]:
num_epochs = 50
for epoch in range(num_epochs): # iterate over the entire dataset 50 times (update 50 times)
  y_hat = model(x, b0, b1) # compute y_hat
  loss = mse_loss(y, y_hat) # compute loss using y_hat

  db0, db1 = gradients(x,y,b0,b1) # compute gradients

  b0 -= lr * db0 # update b0 and b1 by subtracting gradient times learning rate
  b1 -= lr * db1

  # print results every 10 epochs
  if (epoch + 1) % 10 == 0:
    print(f'Epoch {epoch + 1:2d} | Loss = {loss.item():.4f} | b0 = {b0:.4f} | b1 = {b1:.4f}')

Epoch 10 | Loss = 0.0700 | b0 = 0.4329 | b1 = 1.8487
Epoch 20 | Loss = 0.0274 | b0 = 0.3145 | b1 = 1.9254
Epoch 30 | Loss = 0.0166 | b0 = 0.2191 | b1 = 1.9585
Epoch 40 | Loss = 0.0107 | b0 = 0.1486 | b1 = 1.9825
Epoch 50 | Loss = 0.0076 | b0 = 0.0965 | b1 = 2.0002


In [40]:
# compare results to solving it analytically:
X = torch.cat([torch.ones(4,1), x.unsqueeze(1)], dim = 1)
beta_hats = torch.linalg.inv(X.permute(1,0) @ X) @ X.permute(1,0) @ y.unsqueeze(1)
beta_hats
# we should train longer

tensor([[-0.0500],
        [ 2.0500]])

Practice Question 1:
Use the same method, data, and model, but change the loss to be the ridge loss with $\lambda = 0.5$:

$$\mathcal{L}_{\text{Ridge}} = [\frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2] + 0.5\beta_1^2 $$

In [41]:
y = torch.tensor([2.0, 4.1, 6.0, 8.2], dtype = torch.float32) # same simple data
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype = torch.float32)

b0_r = 0
b1_r = 0

def ridge_model(x, b0_r, b1_r): # function for model that gives y_hat using the ridge regression coefficients
  return b0_r + b1_r*x

  # write the updated code here


In [42]:
# parameters should be close to this:
print(torch.linalg.inv(X.permute(1,0) @ X + 2 * torch.tensor([[0,0],[0,1]])) @ X.permute(1,0) @ y.unsqueeze(1))
# you may need to increase the number of epochs etc. (try 200+)


tensor([[1.4143],
        [1.4643]])


Example 2:

Now we will tackle the same problem as Example 1, but using torch's autograd functions.  So we are still using linear regression with MSE loss and a single feature.  The setup is mostly the same, but we want to track the gradient on the parameters.  We no longer need to write our own functions for the gradient, but we do need to instruct pytorch correctly.



In [43]:
y = torch.tensor([2.0, 4.1, 6.0, 8.2], dtype = torch.float32) # simple data (unchanged)
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype = torch.float32)

b0_t = torch.tensor(0.0, requires_grad = True) # set gradient tracking on learnable parameters (changed)
b1_t = torch.tensor(0.0, requires_grad = True)

def model(x, b0_t, b1_t): # function for model that gives y_hat using the coefficients (unchanged)
  return b0_t + b1_t * x

def mse_loss(y, y_hat): # loss function (unchanged)
  errors = y - y_hat
  squared_errors = errors ** 2
  return torch.mean(squared_errors)

# we no longer need a function to compute the gradients, pytorch will do it

lr = 0.1 # same learning rate

num_epochs = 100
for epoch in range(num_epochs):
  # clear gradients before each loop starts (if we don't they will be accumulated every time we use loss.backward())
  b0_t.grad = None
  b1_t.grad = None

  # "forward pass" (inputs to predictions + loss)
  y_hat = model(x, b0_t, b1_t) # compute y_hat
  loss = mse_loss(y, y_hat) # compute loss using y_hat

  # "backward pass" (outputs to gradients)
  loss.backward() # computes gradients automatically on all gradient tracking parameters involved in the loss function

  with torch.no_grad(): # we do not want to track/update gradients for these computations, so we proceed with no_grad() (otherwise it tracks the gradient for every operation etc)
    b0_t -= lr * b0_t.grad # update in direction of negative gradients (use -= here to avoid weird issues with re-assignment and instead update the parameters in memory)
    b1_t -= lr * b1_t.grad

  # print results every 10 epochs
  if (epoch + 1) % 10 == 0:
    print(f'Epoch {epoch + 1:2d} | Loss = {loss.item():.4f} | b0 = {b0_t:.4f} | b1 = {b1_t:.4f}')


Epoch 10 | Loss = 0.0700 | b0 = 0.4329 | b1 = 1.8487
Epoch 20 | Loss = 0.0274 | b0 = 0.3145 | b1 = 1.9254
Epoch 30 | Loss = 0.0166 | b0 = 0.2191 | b1 = 1.9585
Epoch 40 | Loss = 0.0107 | b0 = 0.1486 | b1 = 1.9825
Epoch 50 | Loss = 0.0076 | b0 = 0.0965 | b1 = 2.0002
Epoch 60 | Loss = 0.0058 | b0 = 0.0581 | b1 = 2.0132
Epoch 70 | Loss = 0.0049 | b0 = 0.0298 | b1 = 2.0229
Epoch 80 | Loss = 0.0044 | b0 = 0.0089 | b1 = 2.0300
Epoch 90 | Loss = 0.0041 | b0 = -0.0066 | b1 = 2.0352
Epoch 100 | Loss = 0.0039 | b0 = -0.0179 | b1 = 2.0391


Practice Question 2:
Use the same method, data, and model, but change the loss to be the ridge loss with $\lambda = 0.5$:

$$\mathcal{L}_{\text{Ridge}} = [\frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2] + 0.5\beta_1^2 $$

In [44]:
y = torch.tensor([2.0, 4.1, 6.0, 8.2], dtype = torch.float32) # simple data
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype = torch.float32)

# write updated code here


In [45]:
# parameters should be close to this:
print(torch.linalg.inv(X.permute(1,0) @ X + 2 * torch.tensor([[0,0],[0,1]])) @ X.permute(1,0) @ y.unsqueeze(1))
# you may need to increase the number of epochs etc. (try 200+)

tensor([[1.4143],
        [1.4643]])


Example 3.

We use the same setup as Example 1 and 2, but using almost exclusively Pytorch functions.  The process is similar to Example 2 but we have access to built in functions for most models, losses, etc.

In [46]:
y = torch.tensor([2.0, 4.1, 6.0, 8.2], dtype = torch.float32).unsqueeze(1) # same simple data but we want column dimensions now to plug into the pytorch models
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype = torch.float32).unsqueeze(1)

model = nn.Linear(in_features = 1, out_features = 1) # linear layer that takes in one feature and gives a single output, in this case serving as our simple linear regression y_hat = b0+b1x.
                                                     # this isn't important now, but a lot of models involve several linear layers stacked together in creative ways to add non-linearity or other flexibilities.
criterion = nn.MSELoss() # built-in MSE loss function, standard in ML is to call it the criterion
optimizer = torch.optim.SGD(model.parameters(), lr = 0.1) # we give the optimizer the model parameters (the betas in this case) to update as well as the learning rate

num_epochs = 250
for epoch in range(num_epochs):
  #ensure gradient is 0 before loop starts
  optimizer.zero_grad()
  # forward pass
  y_hat = model(x) # get y_hat
  loss = criterion(y_hat, y) # compute loss, predicted is first argument here

  # backward pass
  loss.backward() # computes gradients as in Example 2
  optimizer.step() # optimizer automatically updates the parameters that we gave it (the betas) using the learning rate etc.
  if (epoch + 1) % 10 == 0:
    print(f'Epoch {epoch + 1:2d} | Loss = {loss.item():.4f} | b0 = {model.bias.item():.4f} | b1 = {model.weight.item():.4f}')


Epoch 10 | Loss = 0.0190 | b0 = 0.1622 | b1 = 1.9577
Epoch 20 | Loss = 0.0084 | b0 = 0.1110 | b1 = 1.9949
Epoch 30 | Loss = 0.0063 | b0 = 0.0689 | b1 = 2.0096
Epoch 40 | Loss = 0.0051 | b0 = 0.0377 | b1 = 2.0202
Epoch 50 | Loss = 0.0045 | b0 = 0.0147 | b1 = 2.0280
Epoch 60 | Loss = 0.0042 | b0 = -0.0022 | b1 = 2.0338
Epoch 70 | Loss = 0.0040 | b0 = -0.0148 | b1 = 2.0380
Epoch 80 | Loss = 0.0039 | b0 = -0.0240 | b1 = 2.0412
Epoch 90 | Loss = 0.0038 | b0 = -0.0308 | b1 = 2.0435
Epoch 100 | Loss = 0.0038 | b0 = -0.0358 | b1 = 2.0452
Epoch 110 | Loss = 0.0038 | b0 = -0.0396 | b1 = 2.0464
Epoch 120 | Loss = 0.0038 | b0 = -0.0423 | b1 = 2.0474
Epoch 130 | Loss = 0.0038 | b0 = -0.0443 | b1 = 2.0481
Epoch 140 | Loss = 0.0038 | b0 = -0.0458 | b1 = 2.0486
Epoch 150 | Loss = 0.0038 | b0 = -0.0469 | b1 = 2.0489
Epoch 160 | Loss = 0.0038 | b0 = -0.0477 | b1 = 2.0492
Epoch 170 | Loss = 0.0038 | b0 = -0.0483 | b1 = 2.0494
Epoch 180 | Loss = 0.0038 | b0 = -0.0488 | b1 = 2.0496
Epoch 190 | Loss = 0.003

In [47]:
# for comparison:
beta_hats
# seems to need about 250 epochs at this learning rate

tensor([[-0.0500],
        [ 2.0500]])

In [48]:
y = torch.tensor([2.0, 4.1, 6.0, 8.2], dtype = torch.float32).unsqueeze(1) # same simple data but we want column dimensions now to plug into the pytorch models
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype = torch.float32).unsqueeze(1)

model = nn.Linear(in_features = 1, out_features = 1) # linear layer that takes in one feature and gives a single output, in this case serving as our simple linear regression y_hat = b0+b1x.
                                                     # this isn't important now, but a lot of models involve several linear layers stacked together in creative ways to add non-linearity or other flexibilities.
criterion = nn.MSELoss() # built-in MSE loss function, standard in ML is to call it the criterion
optimizer = torch.optim.SGD(model.parameters(), lr = 0.1) # we give the optimizer the model parameters (the betas in this case) to update as well as the learning rate

num_epochs = 250
for epoch in range(num_epochs):
  #ensure gradient is 0 before loop starts
  optimizer.zero_grad()
  # forward pass
  y_hat = model(x) # get y_hat
  loss = criterion(y_hat, y) + 0.5 * (model.weight ** 2) # compute loss but add ridge penalty to it (model.weight stores beta1, model.bias stores beta0)

  # backward pass
  loss.backward() # computes gradients as in Example 2
  optimizer.step() # optimizer automatically updates the parameters that we gave it (the betas) using the learning rate etc.
  # print results every 10 epochs
  if (epoch + 1) % 10 == 0:
    print(f'Epoch {epoch + 1:2d} | Loss = {loss.item():.4f} | b0 = {model.bias.item():.4f} | b1 = {model.weight.item():.4f}')

Epoch 10 | Loss = 1.7008 | b0 = 0.6563 | b1 = 1.6377
Epoch 20 | Loss = 1.5624 | b0 = 0.9213 | b1 = 1.6178
Epoch 30 | Loss = 1.5302 | b0 = 1.0865 | b1 = 1.5690
Epoch 40 | Loss = 1.5160 | b0 = 1.1959 | b1 = 1.5343
Epoch 50 | Loss = 1.5097 | b0 = 1.2687 | b1 = 1.5109
Epoch 60 | Loss = 1.5069 | b0 = 1.3173 | b1 = 1.4954
Epoch 70 | Loss = 1.5056 | b0 = 1.3496 | b1 = 1.4850
Epoch 80 | Loss = 1.5051 | b0 = 1.3712 | b1 = 1.4781
Epoch 90 | Loss = 1.5048 | b0 = 1.3856 | b1 = 1.4735
Epoch 100 | Loss = 1.5047 | b0 = 1.3952 | b1 = 1.4704
Epoch 110 | Loss = 1.5047 | b0 = 1.4015 | b1 = 1.4684
Epoch 120 | Loss = 1.5047 | b0 = 1.4058 | b1 = 1.4670
Epoch 130 | Loss = 1.5047 | b0 = 1.4086 | b1 = 1.4661
Epoch 140 | Loss = 1.5046 | b0 = 1.4105 | b1 = 1.4655
Epoch 150 | Loss = 1.5046 | b0 = 1.4118 | b1 = 1.4651
Epoch 160 | Loss = 1.5046 | b0 = 1.4126 | b1 = 1.4648
Epoch 170 | Loss = 1.5046 | b0 = 1.4132 | b1 = 1.4646
Epoch 180 | Loss = 1.5046 | b0 = 1.4135 | b1 = 1.4645
Epoch 190 | Loss = 1.5046 | b0 = 1.41

In [49]:
# parameters should be close to this:
print(torch.linalg.inv(X.permute(1,0) @ X + 2 * torch.tensor([[0,0],[0,1]])) @ X.permute(1,0) @ y)

tensor([[1.4143],
        [1.4643]])


Conceptual Questions:

1. Why do we move in the direction of the negative gradient?

2. What can tend to happen if the learning rate is too large?

3. What can tend to happen if the learning rate is too small?

4. How can we potentially tell if the learning rate is too small or too large?

5. Why does gradient descent eventually stop improving the model/loss? (If we set the epochs to an absurd number etc.)

6. Does gradient descent always eventually find a $\textbf{global}$ minimum?

7. When the gradient is $0$, what does it mean?

8. What criteria should we look for to know when to stop training?

9. What do you think makes more complex models than linear regression harder to optimize (even when gradient descent is used to train both)?

10. Why do we have to keep re-computing the gradients at every step?

11. Does the size of a feature influence its gradient? (Would weight in tons generate different gradients than weight in pounds of the same observations?)

12. Suppose we have two features.  What happens for fitting with gradient descent if one of them has large gradients and the other very small ones?  

13. What happens if we set the ridge penalty $\lambda$ from $0.5$ to $0$?

14. Relative to MSE, what differences in the parameters do we expect from ridge for $\lambda>0$.

15. If we used two features for ridge, why would we want them to both be scaled similarly?  (IE ensure both of them range from 0 to 1 instead of one of them ranging from 0-100 and the other from 0-10).

16. In what ways might our starting values for the parameters matter?
